# Plot results for offline prompting in the "farm" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [2]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys
import json

In [23]:
results_folder = Path("generated_adaptations/farm")
pd.options.display.float_format = "{:,.1f}".format

## Run experiments

In [28]:
prompts = {
    "default": "generated_adaptations/prompts/farm_strategy.md",
    "state": "generated_adaptations/prompts/farm_strategy_state.md",
}
variants = {
    "default": [],
    "notest": ["--retries_test=0"],
    "constraints": []
}
llms = {
    # "5nanolow": "gpt-5-nano-2025-08-07,reasoning_effort=low",
    "5nano": "gpt-5-nano-2025-08-07",
    "5mini": "gpt-5-mini-2025-08-07",
    # "5": "gpt-5-2025-08-07",
}
repeats = 2
start = 1

In [29]:
# import generated_adaptations.generator as generator

In [30]:
for prompt_name, prompt in prompts.items():
    for variant, args in variants.items():
        for llm_name, llm in llms.items():
            for repeat in range(start, start + repeats):
                folder_name = f"{llm_name}_{repeat:02d}"
                print(f"\n{variant}_{prompt_name}/{folder_name}\n")
                folder = results_folder / f"{variant}_{prompt_name}" / folder_name
                folder.mkdir(parents=True, exist_ok=True)
                shutil.copy(prompt, folder / "01_01_user.md")
                cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}", f"--llm={llm}", *args]
                result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
                print(result.stdout)
                print(result.stderr)
                # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


default_default/5nano_01

Loaded 2 messages from generated_adaptations\farm\default_default\5nano_01.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\farm\default_default\5nano_01\code_01_02.py'.
TOKENS USED:
Input: 1010, Output: 8070 (reasoning: 7040)
Response time (seconds): 43.5

Running tests: pytest generated_adaptations/tests -q --tb=short -rfExXpP --show-capture=no --color=no --example=farm --adaptation_name=5nano_01/code_01_02 --variant=default_default
Test exit code: 0
Running simulation for 'generated_adaptations\farm\default_default\5nano_01\code_01_02.py'.
  Run #1/3: python main.py farm/configs/default.yaml generated_adaptations/configs/generated.yaml farm/configs/config_no_battery.yaml DSL/drones.yaml --extra_config={"name": "default_default/5nano_01/code_01_02", "log_dir.append": "/default_default/5nano_01/code_01_02", "adaptation_name": "generated_adaptations.farm.default_default.5nano_01.code_01_02.

## Results

In [31]:
folders = list(results_folder.glob("*/*"))
# folders = list(results_folder.glob("41mini_*"))
# folders = list(results_folder.glob("*/5nanolow*"))
# folders = [results_folder / "41mini"]

folders = [f for f in folders if "mock" not in f.parent.stem]
print([(f.parent.stem, f.stem) for f in folders])

[('constraints_default', '5mini_01'), ('constraints_default', '5mini_02'), ('constraints_default', '5nanolow_01'), ('constraints_default', '5nanolow_02'), ('constraints_default', '5nano_01'), ('constraints_default', '5nano_02'), ('constraints_state', '5mini_01'), ('constraints_state', '5mini_02'), ('constraints_state', '5nanolow_01'), ('constraints_state', '5nanolow_02'), ('constraints_state', '5nano_01'), ('constraints_state', '5nano_02'), ('default_default', '5mini_01'), ('default_default', '5mini_02'), ('default_default', '5nanolow_01'), ('default_default', '5nanolow_02'), ('default_default', '5nano_01'), ('default_default', '5nano_02'), ('default_state', '5mini_01'), ('default_state', '5mini_02'), ('default_state', '5nanolow_01'), ('default_state', '5nanolow_02'), ('default_state', '5nano_01'), ('default_state', '5nano_02'), ('notest_default', '5mini_01'), ('notest_default', '5mini_02'), ('notest_default', '5nanolow_01'), ('notest_default', '5nanolow_02'), ('notest_default', '5nano

In [32]:
summary = pd.DataFrame(columns=["llm", "params", "repeat"])
best = pd.DataFrame(columns=["llm", "params", "repeat"])
for folder in folders:
    llm, repeat = folder.stem.split("_")
    params = folder.parent.stem
    summary.loc[len(summary), ["llm", "params", "repeat"]] = [llm, params, repeat]
    best.loc[len(best), ["llm", "params", "repeat"]] = [llm, params, repeat]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "pass"
        elif "_simulation_result" in name:  # older: "_simulation_result.txt"
            code = name.removesuffix("_simulation_result")
            with open(file, "r") as f:
                result = f.read().strip()
                result = result.split("\n")[0].split(": ")[-1]  # note that this is specific for the farm example
            summary.loc[len(summary) - 1, code + "_result"] = result
            best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
        else:
            print(f"Unknown file: {file}")
        for file in (folder / "results").glob("*.json"):  # newer: "_simulation_result.json"
            name = file.stem.removeprefix("code_")
            if "_simulation_result" in name:
                code = name.removesuffix("_simulation_result")
                results = json.load(open(file))
                result = results["damage"]
                summary.loc[len(summary) - 1, code + "_result"] = result
                best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
            else:
                print(f"Unknown file: {file}")
summary.fillna("", inplace=True)
best.fillna("", inplace=True)

C:\Users\micha\AppData\Local\Temp\ipykernel_7540\2306563932.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary.fillna("", inplace=True)
C:\Users\micha\AppData\Local\Temp\ipykernel_7540\2306563932.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  best.fillna("", inplace=True)


In [33]:
summary.set_index(["llm", "params", "repeat"], inplace=True)
best.set_index(["llm", "params", "repeat"], inplace=True)

In [34]:
summary = summary.reindex(sorted(summary.columns), axis=1)
best = best.reindex(sorted(best.columns), axis=1)

In [35]:
summary

01_02_result 01_02_test 01_04_result  \
llm      params              repeat                                        
5mini    constraints_default 01            123.3       fail         57.7   
                             02            123.0       fail                
5nanolow constraints_default 01                        fail        120.7   
                             02            120.7       fail        279.0   
5nano    constraints_default 01            250.3       fail        114.7   
                             02            123.3       fail         66.3   
5mini    constraints_state   01            123.0       fail                
                             02            118.3       fail                
5nanolow constraints_state   01            123.3       fail        123.0   
                             02            123.0       fail        130.3   
5nano    constraints_state   01            123.0       fail         83.7   
                             02            135.7       pass                
5mini    default_default     01            119.7       pass                
                             02            147.0       pass                
5nanolow default_default     01             56.7       pass                
                             02            120.7       fail        120.7   
5nano    default_default     01            123.3       pass                
                             02            123.3       fail        123.0   
5mini    default_state       01            111.7       pass                
                             02            123.0       pass                
5nanolow default_state       01                        fail        286.0   
                             02            241.7       fail        120.7   
5nano    default_state       01            123.0       pass                
                             02            171.7       fail        144.7   
5mini    notest_default      01            123.0       pass                
                             02            119.7       pass                
5nanolow notest_default      01            120.7       pass                
                             02            175.3       fail                
5nano    notest_default      01            123.0       pass                
                             02            120.7       pass                
5mini    notest_state        01            123.0       pass                
                             02            122.0       pass                
5nanolow notest_state        01            278.0       fail                
                             02            154.3       pass                
5nano    notest_state        01            288.0       pass                
                             02            123.3       pass                

                                    01_04_test 01_06_result 01_06_test  \
llm      params              repeat                                      
5mini    constraints_default 01           pass                           
                             02                                          
5nanolow constraints_default 01           fail        143.7       pass   
                             02           fail        279.0       fail   
5nano    constraints_default 01           fail         54.3       pass   
                             02           pass                           
5mini    constraints_state   01           fail                           
                             02           fail                           
5nanolow constraints_state   01           fail        128.7       pass   
                             02           fail        255.0       fail   
5nano    constraints_state   01           fail        255.0       fail   
                             02                                          
5mini    default_default     01                                          
                             02      

In [36]:
best

01_result 01_test 02_result 02_test  \
llm      params              repeat                                        
5mini    constraints_default 01           57.7    pass      49.7    pass   
                             02          123.0    fail                     
5nanolow constraints_default 01          143.7    pass     143.7    pass   
                             02          279.0    fail     279.0    fail   
5nano    constraints_default 01           54.3    pass      54.3    pass   
                             02           66.3    pass      66.3    pass   
5mini    constraints_state   01          123.0    fail                     
                             02          118.3    fail                     
5nanolow constraints_state   01          128.7    pass     128.7    pass   
                             02           66.3    pass      65.3    pass   
5nano    constraints_state   01           47.0    pass     123.0    fail   
                             02          135.7    pass     332.3    fail   
5mini    default_default     01          119.7    pass      67.7    pass   
                             02          147.0    pass     109.0    pass   
5nanolow default_default     01           56.7    pass     187.0    pass   
                             02          120.7    pass     123.3    pass   
5nano    default_default     01          123.3    pass      60.7    pass   
                             02          123.0    pass     123.0    pass   
5mini    default_state       01          111.7    pass                     
                             02          123.0    pass      54.3    pass   
5nanolow default_state       01          286.0    pass     286.7    pass   
                             02          120.7    pass     120.7    pass   
5nano    default_state       01          123.0    pass     121.7    pass   
                             02          144.7    pass      60.7    pass   
5mini    notest_default      01          123.0    pass      50.0    pass   
                             02          119.7    pass      57.0    pass   
5nanolow notest_default      01          120.7    pass     241.7    fail   
                             02          175.3    fail     120.7    pass   
5nano    notest_default      01          123.0    pass      60.7    fail   
                             02          120.7    pass     312.7    pass   
5mini    notest_state        01          123.0    pass      41.0    pass   
                             02          122.0    pass                     
5nanolow notest_state        01          278.0    fail     241.7    fail   
                             02          154.3    pass     108.0    pass   
5nano    notest_state        01          288.0    pass     123.0    pass   
                             02          123.3    pass     299.3    fail   

                                    03_result 03_test  
llm      params              repeat                    
5mini    constraints_default 01         145.0    pass  
                             02                        
5nanolow constraints_default 01         143.7    pass  
                             02         274.3    fail  
5nano    constraints_default 01          54.3    pass  
                             02          66.3    pass  
5mini    constraints_state   01                        
                             02                        
5nanolow constraints_state   01         128.7    pass  
                             02          65.3    pass  
5nano    constraints_state   01         255.0    fail  
                             02         352.0    fail  
5mini    default_default     01         164.3    pass  
                             02         145.0    pass  
5nanolow default_default     01         126.3    pass  
                             02         123.3    pass  
5nano    default_default     01          60.7    pass  
                             02                        
5mini    default_state      